# Chapter 4: ZCI of Bray-Curtis Ordination on Community Composition

**Objective**: 

Quantifing the pattern of Community Composition of multivariate by nature in each identified community typology.

**Reasons/Motivations**:

Toxonomically rich assemblages can better represent the community responses because they averge out individual-level variability and capture broader ecological patterns. Meanwhile it reduces the contamination effects on community composition into a single index, easier to interpret and to compare sites.

**How**:

In each community type, the ZCI is calculated by the following steps.

Firstly, setting two ecological endpints as reference and degraded community pattern respectively,
then preserve them in original data matrix with $X_{(m+2) \times (m+2)}$, calculate the Bray-Curtis dissimilarity matrix $D_{(m+2) \times (m+2)}$, and then apply Non-metric Multidimensional Scaling (NMDS) to decompose the $D_{(m+2) \times (m+2)}$ into a $k$ low-dimensional ordination space $Y_{(m+2) \times k}$. Finally, calculate the ZCI by processing the coordinations of sites in $Y_{(m+2) \times k}$.

Specifically, the computation objects are as follows:

$$
X_{(m+2) \times (m+2)} \rightarrow D_{(m+2) \times (m+2)} \rightarrow Y_{(m+2) \times k} \rightarrow ZCI_{(m+2) \times 1}
$$

**Results**:

In each community type, every site has its ZCI value that quantifies its community pattern in the ordination space, such value reflects its similarity  to the chosen reference and degraded endpoints in term of community composition pattern.


## Framework 1: Taxa Bray-Curtis Dissimilarity Matrix with Poles (Endpoints)

On the taxa data, Bray-Curtis dissimilarity can quantify the distance between site $i$ and site $j$ with their abundance data of taxon $k$ meaningfully and aviod the **double-zero problem**.

**Question/Motivation**:

Euclidean distance (dissimilarity) is not suitable for measureing difference of two sites in community composition,
especially the **shared absence** of some taxa in both sites does not mean they are ecologically similar.
It raises the need of a measure that handles the double-zero problem and focuses on the observed shared abundance and abundance differences among sites.

**Methods and Steps**:

Bray-Curtis is the dissimilarity measure to quantify the dissimilarity in terms of taxa composition between two sites. It focuses on shared abundance and abundance differences, not on shared absences. It tells out of the total abundance observed (first) across the two sites, how much is not shared (second) or needs to be redistributed to make them similar?

Specifically, the Bray-Curtis dissimilarity between site $i$ and site $j$ in terms of $k$ taxa is calculated as:

$$
BC_{ij} = \frac{\sum_{k=1}^{16} |x_{ik} - x_{jk}|}{\sum_{k=1}^{16} (x_{ik} + x_{jk})}
$$

| $BC_{ij}$ value | Meaning for sites $i$ and $j$ |
|:-----------------:|:--------:|
| 0 | identical composition |
| close to 0 | very similar composition |
| close to 1 | very different composition |
| 1 | no shared taxa/abundance |

Bray-Curtis can be applied on the proportional octave-transformed taxa data, which is the raw taxa format in our study. Meanwhile, the bounded values between 0 and 1 scale is useful for ordination and ecological interpretation.
Except the soly similarities between sites, setting two endpoints as reference and degraded community patterns can assign ecological meanings to the distance direction, which augments the taxa matrix from $m$ sites to $m + 2$.

**Results**:

Applying Bray-Curtis dissimilarity for all pairs of $(m + 2)$ sites, it produces a Bray-Curtis dissimilarity matrix $D_{(m+2) \times (m+2)}$ of all sites.

$$
\mathbf{D}_{(m+2) \times (m+2)}
=

\begin{bmatrix}
BC_{m \times m} & BC_{m \times 2}\\
BC_{2 \times m} & BC_{2 \times 2} \\
\end{bmatrix}

=
\begin{bmatrix}
BC_{11} & BC_{12} & \cdots  & BC_{1E_{ref}} & BC_{1E_{deg}} \\
BC_{21} & BC_{22} & \cdots  & BC_{2E_{ref}} & BC_{2E_{deg}} \\
\vdots & \vdots & \ddots & \vdots & \vdots \\
BC_{E_{ref}1} & BC_{E_{ref}2} & \cdots  & BC_{E_{ref}E_{ref}} & BC_{E_{ref}E_{deg}} \\
BC_{E_{deg}1} & BC_{E_{deg}2} & \cdots & BC_{E_{deg}E_{ref}} & BC_{E_{deg}E_{deg}} \\
\end{bmatrix}
$$


Therefore, the transformation from original taxa abundance data with two endpoints to the Bray-Curtis dissimilarity matrix can be summarized as:

$$
\mathbf{X}_{m \times p}
\xrightarrow{\text{Augment}}
\mathbf{X}_{(m+2) \times p}
\xrightarrow{\text{Bray-Curtis}}
\mathbf{D}_{(m+2) \times (m+2)}
$$


### Computation Workflow

**Inputs**:

1. metadata: taxa data $\mathbf{T}$
2. artifacts: 
    - **table** : posterior probabilities of community types of sites
      - rows: $(\text{Integrated Code}, P_{i, posterior} = C_1, P_{i, posterior} = C_2, P_{i, posterior} = C_3)$
      - shape: $(310, 4)$
    - **table**: contamination scores of sites
      - rows: $(\text{Integrated Code}, PC_1 \text{ scores}, \ldots, PC_5 \text{ scores}, f_{sum}(), f_{max}())$
      - shape: $(310, 8)$

**Process**:

1. **Transformations**
   - **None**: $\mathbf{T}$ is already in octave format.

2. **Bridging tools**
   - **Boolean filter for posterior probabilities**: filters sites with confident community-type assignment.
   - **Rank-based subset selector**: selects sites according to ranked values of contamination scores.

3. **Modeling tools**
   - **Averaging operator**: computes the centroid or mean composition vector from selected samples.
   - **Bray-Curtis dissimilarity**: computes pairwise compositional dissimilarities among sites and endpoints.

4. **Interpreters**
   - **Bray-Curtis dissimilarity matrix**: a $D_{(m+2) \times (m+2)}$ matrix quantifying compositional dissimilarities among all pairs of real sites and the two endpoints.


**Demo Outputs**:


$$
\mathbf{D}_{BC}^{(m+2)\times(m+2)}
=
\begin{bmatrix}
0 
& d_{12} 
& \cdots 
& d_{1m} 
& d_{1,\mathrm{ref}} 
& d_{1,\mathrm{deg}} \\

d_{21} 
& 0 
& \cdots 
& d_{2m} 
& d_{2,\mathrm{ref}} 
& d_{2,\mathrm{deg}} \\

\vdots 
& \vdots 
& \ddots 
& \vdots 
& \vdots 
& \vdots \\

d_{m1} 
& d_{m2} 
& \cdots 
& 0 
& d_{m,\mathrm{ref}} 
& d_{m,\mathrm{deg}} \\

d_{\mathrm{ref},1} 
& d_{\mathrm{ref},2} 
& \cdots 
& d_{\mathrm{ref},m} 
& 0 
& d_{\mathrm{ref},\mathrm{deg}} \\

d_{\mathrm{deg},1} 
& d_{\mathrm{deg},2} 
& \cdots 
& d_{\mathrm{deg},m} 
& d_{\mathrm{deg},\mathrm{ref}} 
& 0
\end{bmatrix}
$$

## Framework 2: NMDS Ordination of Bray-Curtis Dissimilarity Matrix

**Question/Motivation**:

**Methods/Steps**:

**Results**: